In [ ]:
# # Chapter 2: Pre-Model Workflow and Data Preprocessing
# This notebook provides recipes for preprocessing data using scikit-learn:
# - Handling missing values
# - Scaling techniques
# - Encoding categorical variables
# - ColumnTransformers
# - Pipelines
# - Feature engineering
# - Feature selection
# - A fully integrated pipeline applied to California Housing dataset


In [1]:
# ## Handling Missing Data
import numpy as np
import pandas as pd

np.random.seed(2024)
n_samples = 20
n_features = 10

data = {f"Feature{i+1}": np.random.uniform(0, 100, n_samples)
        for i in range(n_features)}

df = pd.DataFrame(data)

# Introduce missing values (~20%)
for col in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, col] = np.nan

df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


In [2]:
### SimpleImputer
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_df = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)
imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,57.822964,22.966899,21.049022,57.358544,14.302591
7,52.558605,53.864661,89.538184,69.451294,46.055883,47.885551,51.145011,33.620401,99.685711,6.683138
8,47.384570,53.864661,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.897507,31.142866,28.865545,57.822964,41.004459,41.460336,50.236236,48.400044


In [3]:
### KNNImputer
from sklearn.impute import KNNImputer

knn = KNNImputer(n_neighbors=2)
knn_imputed_df = pd.DataFrame(knn.fit_transform(df), columns=df.columns)
knn_imputed_df


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,40.482911,22.966899,21.049022,57.358544,14.302591
7,47.629715,77.843853,89.538184,69.451294,32.753328,47.885551,55.683434,33.620401,99.685711,6.683138
8,47.384570,35.180639,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,29.188271,31.142866,28.865545,59.674651,41.004459,41.460336,50.236236,35.000820


In [4]:
### IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iter_imputer = IterativeImputer()
iterative_imputed_df = pd.DataFrame(
    iter_imputer.fit_transform(df),
    columns=df.columns
)
iterative_imputed_df


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,55.923578,22.966899,21.049022,57.358544,14.302591
7,52.558131,53.942885,89.538184,69.451294,46.070477,47.885551,50.969808,33.620401,99.685711,6.683138
8,47.384570,54.204978,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.882614,31.142866,28.865545,55.978300,41.004459,41.460336,50.236236,48.429901


### 2. Scaling Techniques


In [5]:
# ## Scaling Techniques
### StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_df = pd.DataFrame(
    scaler.fit_transform(iterative_imputed_df),
    columns=iterative_imputed_df.columns
)
scaled_df


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904
6,0.878636,1.094467,-0.091017,0.042422,-1.929442,-0.075466,-1.302124,-1.273574,-0.110399,-1.133636
7,-0.000135,0.001649,1.255789,0.639061,0.000428,-0.388327,-0.007723,-0.734020,1.640810,-1.386962
8,-0.225584,0.012729,-0.935710,1.210940,1.323678,0.147955,-1.015275,0.653400,0.040472,-0.476999
9,-0.336923,-0.665377,-0.000435,-1.050256,-0.832163,-0.073336,-0.468360,-0.397536,-0.405072,0.001002


In [6]:
### MinMaxScaler
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler()
minmax_scaled_df = pd.DataFrame(
    mm.fit_transform(iterative_imputed_df),
    columns=iterative_imputed_df.columns
)
minmax_scaled_df


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710
6,0.751199,0.811657,0.468490,0.611621,0.000000,0.524114,0.070723,0.218016,0.567681,0.107208
7,0.537276,0.512946,0.888472,0.775311,0.518428,0.431454,0.432317,0.353891,1.000000,0.027004
8,0.482394,0.515975,0.205085,0.932208,0.873897,0.590284,0.150855,0.703283,0.604927,0.315101
9,0.455290,0.330621,0.496737,0.311840,0.294766,0.524745,0.303637,0.438627,0.494936,0.466437


In [7]:
### Normalizer
from sklearn.preprocessing import Normalizer

norm = Normalizer()
normalized_df = pd.DataFrame(
    norm.fit_transform(iterative_imputed_df),
    columns=iterative_imputed_df.columns
)
normalized_df


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177
6,0.460521,0.505281,0.318139,0.354119,0.039204,0.354133,0.145437,0.133292,0.363220,0.090570
7,0.274582,0.281816,0.467778,0.362837,0.240688,0.250170,0.266284,0.175644,0.520792,0.034915
8,0.265283,0.303467,0.143277,0.461427,0.411012,0.345225,0.163323,0.369203,0.341538,0.190645
9,0.321287,0.273524,0.379002,0.223196,0.206875,0.401188,0.293873,0.297140,0.360036,0.347090


### 3. Encoding Categorical Variables


In [8]:
# ## Encoding Categorical Variables
### Create categorical dataset
np.random.seed(2024)
categories = ["A", "B", "C", "D"]

categorical_data = pd.DataFrame({
    "Department": np.random.choice(categories, 20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], 20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], 20),
})
categorical_data


,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


In [9]:
### OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
ohe_df = pd.DataFrame(
    ohe.fit_transform(categorical_data),
    columns=ohe.get_feature_names_out()
)
ohe_df


,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [10]:
### LabelEncoder
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
label_encoded_df = pd.DataFrame({
    f"{col}_encoded": le.fit_transform(categorical_data[col])
    for col in categorical_data.columns
})
label_encoded_df


,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1
6,2,1,3
7,3,1,1
8,1,0,1
9,3,1,0


### 4. ColumnTransformer


In [11]:
### Create mixed data
from sklearn.compose import ColumnTransformer

np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, 20),
    "Salary": np.round(np.random.normal(60000, 15000, 20), 2),
    "Experience": np.random.randint(1, 20, 20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], 20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], 20),
})
mixed_data


,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


In [12]:
### Apply ColumnTransformer
column_tf = ColumnTransformer(
    [
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="drop"
)

transformed = column_tf.fit_transform(mixed_data)

numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
cat_cols = column_tf.named_transformers_["cat"].get_feature_names_out(
    ["Department", "Position"]
)

transformed_df = pd.DataFrame(transformed, columns=numeric_cols + list(cat_cols))
transformed_df


,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,0.949309,-0.895798,0.701364,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,-0.897596,-0.991489,1.262454,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8,-1.488606,-0.580596,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.727681,1.744195,0.701364,0.0,0.0,0.0,1.0,0.0,0.0,1.0


### 5. Pipelines


In [13]:
# ## Introduction to Pipelines
### Build simple preprocessing pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X = transformed_df.copy()
y = np.random.randn(len(X))   # dummy target for demonstration

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

X_train_trans = pd.DataFrame(
    pipe.fit_transform(X_train),
    columns=X_train.columns
)
X_test_trans = pd.DataFrame(
    pipe.transform(X_test),
    columns=X_test.columns
)


In [14]:
### Visualize the pipeline
from sklearn import set_config
set_config(display="diagram")
pipe


Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

### 6. Feature Engineering


In [15]:
### Polynomial features
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_df = pd.DataFrame(
    poly.fit_transform(X_train_trans),
    columns=poly.get_feature_names_out(X_train_trans.columns)
)
poly_df


,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Department_Sales Position_Senior,Position_Junior^2,Position_Junior Position_Manager,Position_Junior Position_Senior,Position_Manager^2,Position_Manager Position_Senior,Position_Senior^2
0,1.0,0.834298,-0.867451,-1.559779,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,-1.0,...,0.600000,0.447214,0.774597,-1.341641,0.333333,0.577350,-1.000000,1.0,-1.732051,3.000000
1,1.0,-0.689202,1.282818,-0.150946,-0.577350,2.081666,-0.480384,-0.774597,-0.577350,1.0,...,0.600000,0.447214,-0.774597,0.447214,0.333333,-0.577350,0.333333,1.0,-0.577350,0.333333
2,1.0,0.544107,-1.173357,-1.761041,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,1.0,...,0.600000,0.447214,-0.774597,0.447214,0.333333,-0.577350,0.333333,1.0,-0.577350,0.333333
3,1.0,-1.342131,0.821525,-0.150946,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,1.666667,-0.745356,-1.290994,2.236068,0.333333,0.577350,-1.000000,1.0,-1.732051,3.000000
4,1.0,1.414679,0.611115,0.654101,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,1.0,...,1.666667,-0.745356,1.290994,-0.745356,0.333333,-0.577350,0.333333,1.0,-0.577350,0.333333
5,1.0,-1.414679,-1.479801,1.257887,1.732051,-0.480384,-0.480384,-0.774597,-0.577350,1.0,...,0.600000,0.447214,-0.774597,0.447214,0.333333,-0.577350,0.333333,1.0,-0.577350,0.333333
6,1.0,1.269583,0.576819,0.452839,1.732051,-0.480384,-0.480384,-0.774597,1.732051,-1.0,...,0.600000,-1.341641,0.774597,0.447214,3.000000,-1.732051,-1.000000,1.0,0.577350,0.333333
7,1.0,0.544107,-1.475450,-0.553470,-0.577350,-0.480384,2.081666,-0.774597,1.732051,-1.0,...,0.600000,-1.341641,0.774597,0.447214,3.000000,-1.732051,-1.000000,1.0,0.577350,0.333333
8,1.0,0.906845,1.060444,1.459148,-0.577350,-0.480384,-0.480384,1.290994,1.732051,-1.0,...,1.666667,2.236068,-1.290994,-0.745356,3.000000,-1.732051,-1.000000,1.0,0.577350,0.333333
9,1.0,0.906845,1.664674,0.855363,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,1.666667,-0.745356,-1.290994,2.236068,0.333333,0.577350,-1.000000,1.0,-1.732051,3.000000


In [16]:
### KBinsDiscretizer
from sklearn.preprocessing import KBinsDiscretizer

kb = KBinsDiscretizer(n_bins=3, encode="ordinal")
binned_df = pd.DataFrame(
    kb.fit_transform(X_train_trans),
    columns=X_train_trans.columns
)
binned_df


c:\Users\irpra\miniconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 3 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\irpra\miniconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 4 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\irpra\miniconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 5 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\irpra\miniconda3\Lib\site-packages\sklearn\preprocessing\_discretization.py:307: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 6 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\irpra\miniconda3\Lib\site-packages\sklearn\preprocessing\_d

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 7. Feature Selection


In [17]:
### RFE
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(LinearRegression(), n_features_to_select=1)
rfe.fit(X_train_trans, y_train)
rfe.ranking_


array([10,  3,  4,  7,  9,  8,  1,  6,  5,  2])

In [18]:
### SelectFromModel
from sklearn.feature_selection import SelectFromModel

sel = SelectFromModel(LinearRegression(), threshold="mean")
sel.fit(X_train_trans, y_train)

selected = X_train_trans.columns[sel.get_support()]
selected


Index(['Salary_scaled', 'Experience_scaled', 'Department_IT',
       'Department_Sales', 'Position_Senior'],
      dtype='object')

### 8. **Fully Integrated Pipeline: California Housing Dataset**

In [19]:
# # Practical Exercise: Full Preprocessing Pipeline on California Housing
### Load data & split
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

In [ ]:
### Build comprehensive pipeline
from sklearn.metrics import mean_squared_error

full_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=2024))
])

full_pipeline.fit(X_train, y_train)
preds = full_pipeline.predict(X_test)

mse = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
rmse


ImportError: cannot import name 'RandomForestRegressor' from 'sklearn.linear_model' (c:\Users\irpra\miniconda3\Lib\site-packages\sklearn\linear_model\__init__.py)


# 📚 Recommended Resources

### **scikit-learn official**

* Documentation: [https://scikit-learn.org/stable/documentation.html](https://scikit-learn.org/stable/documentation.html)
* User guide: [https://scikit-learn.org/stable/user_guide.html](https://scikit-learn.org/stable/user_guide.html)
* Pipeline tutorial: [https://scikit-learn.org/stable/modules/compose.html](https://scikit-learn.org/stable/modules/compose.html)

### **Deep learning + preprocessing**

* Géron, *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow*
* Raschka, *Machine Learning with PyTorch and Scikit-Learn*

### **Feature engineering**

* *Kaggle Feature Engineering Course*
* *Feature Engineering for Machine Learning* (Alice Zheng)
